# CDMX clustered high-risk polygons: household boundary counts

This notebook creates local clusters of high-risk slope-instability polygons and counts cadastre household-proxy points just inside and outside each cluster boundary. It is a screening tool for identifying plausible local study areas; it does not decide whether a boundary is physically comparable.

Each output row is one cluster under one clustering-gap scenario. Constituent polygons are dissolved before counting, so a household is counted at most once within a cluster. Outside zones from *different* clusters are intentionally allowed to overlap and a household may therefore appear in both cluster rows.

In [2]:
%pip -q install geopandas pyogrio shapely pyproj rtree

from pathlib import Path
import re
import unicodedata
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.ops import unary_union
from google.colab import drive

DRIVE_RAW = '/content/drive/MyDrive/Dissertation/data/raw'
WORK = Path('/content/drive/MyDrive/Dissertation/data/output')
RAW = Path(DRIVE_RAW)
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive', force_remount=False)
if not RAW.exists():
    raise FileNotFoundError(f'Raw-data folder not found: {RAW}')
if not WORK.exists():
    raise FileNotFoundError(f'Create the output folder in Drive first: {WORK}')

METRIC_CRS = 'EPSG:32614'
RISK_FIELD = 'INTENSIDAD'
DISTRICT_FIELD = 'nomgeo'
RISK_CATEGORY_ORDER = ['baja', 'media', 'alta']
N_HIGHEST_RISK_CATEGORIES = 1
CADASTRE_TOKENS = {'aob': 'ALVARO_OBREGON', 'cuj': 'CUAJIMALPA', 'gam': 'GUSTAVO_A_MADERO', 'izp': 'IZTAPALAPA'}
DISTRICT_ALIASES = {
    'aob': {'ALVARO OBREGON'},
    'cuj': {'CUAJIMALPA', 'CUAJIMALPA DE MORELOS'},
    'gam': {'GUSTAVO A MADERO'},
    'izp': {'IZTAPALAPA'},
}


## Parameters

`CLUSTER_DISTANCE_METHOD` controls the distance measured between candidate polygons. `perimeter` uses the minimum edge-to-edge distance and is generally the more geographically literal option. `centroid` uses the distance between representative centroids and is included as a sensitivity alternative.

`INSIDE_MODE = 'near_boundary'` counts households in the inner 20 m boundary strip; switch it to `all_inside` to count every household inside the dissolved cluster.

In [3]:
CLUSTER_GAPS_M = [25, 50, 100, 200]
CLUSTER_DISTANCE_METHOD = 'perimeter'  # choose 'perimeter' or 'centroid'
OUTSIDE_BAND_M = 20
INSIDE_MODE = 'near_boundary'  # choose 'near_boundary' or 'all_inside'
INSIDE_BAND_M = 20

if CLUSTER_DISTANCE_METHOD not in {'perimeter', 'centroid'}:
    raise ValueError("CLUSTER_DISTANCE_METHOD must be 'perimeter' or 'centroid'.")
if INSIDE_MODE not in {'near_boundary', 'all_inside'}:
    raise ValueError("INSIDE_MODE must be 'near_boundary' or 'all_inside'.")
if any(gap <= 0 for gap in CLUSTER_GAPS_M) or OUTSIDE_BAND_M <= 0 or INSIDE_BAND_M <= 0:
    raise ValueError('All distance parameters must be positive metres.')


## Load and standardize inputs

The helper functions deliberately fail when the raw-data folder contains zero or multiple plausible matches. This avoids silently using a different layer after files are renamed or duplicated.

In [4]:
def norm(value):
    """Return a lowercase, accent-free token suitable for filename and name matching."""
    value = unicodedata.normalize('NFKD', str(value)).encode('ascii', 'ignore').decode('ascii').lower()
    return re.sub(r'[^a-z0-9]+', '_', value).strip('_')

raw_files = [path for path in RAW.rglob('*') if path.is_file()]

def matches(*tokens, extensions={'.geojson', '.json', '.zip'}):
    """Find spatial input files whose normalized names contain every requested token."""
    normalized_tokens = [norm(token) for token in tokens]
    def contains(name, token):
        return token in name or token.replace('_', '') in name.replace('_', '')
    return [path for path in raw_files if path.suffix.lower() in extensions and all(contains(norm(path.name), token) for token in normalized_tokens)]

def one(label, *tokens):
    """Return the unique matching input file or raise an explicit diagnostic error."""
    found = matches(*tokens)
    if len(found) != 1:
        raise FileNotFoundError(f'{label}: expected one matching file, found {found}')
    return found[0]

def read(path):
    """Read either a standalone spatial file or a zipped spatial dataset."""
    return gpd.read_file(f'zip:///{path}' if path.suffix.lower() == '.zip' else path)

def clean(frame, label):
    """Validate non-empty geometries and reproject them to the common metre-based CRS."""
    if frame.crs is None:
        raise ValueError(f'{label} has no CRS.')
    frame = frame.to_crs(METRIC_CRS).copy()
    frame = frame[frame.geometry.notna() & ~frame.geometry.is_empty]
    frame['geometry'] = frame.geometry.make_valid()
    return frame[frame.geometry.notna() & ~frame.geometry.is_empty]

risk_file = one('Risk zones', 'inestabilidad', 'laderas')
district_matches = matches('alcald')
if len(district_matches) != 1:
    raise FileNotFoundError(f'District file (alcaldias/alcadias): expected one match, found {district_matches}')
district_file = district_matches[0]
cadastre_files = {code: one(f'Cadastre {code}', 'catastro2021', token) for code, token in CADASTRE_TOKENS.items()}


## Create high-risk polygons and household-proxy points

Risk polygons are clipped to the four selected alcaldías before clustering. A cadastre parcel is represented by one point, so it cannot be counted twice because its source geometry crosses a boundary.

In [5]:
risk = clean(read(risk_file), risk_file.name).explode(index_parts=False, ignore_index=True)
risk = risk[risk.geometry.geom_type == 'Polygon'][[RISK_FIELD, 'geometry']].rename(columns={RISK_FIELD: 'risk_category'}).copy()
risk['risk_category'] = risk['risk_category'].astype(str).map(norm)
selected_categories = set(norm(value) for value in RISK_CATEGORY_ORDER[-N_HIGHEST_RISK_CATEGORIES:])
risk = risk[risk.risk_category.isin(selected_categories)].copy()

districts = clean(read(district_file), district_file.name)[[DISTRICT_FIELD, 'geometry']].rename(columns={DISTRICT_FIELD: 'district_name'})
def district_code(name):
    """Map an alcaldía name from the boundary layer to this study's short code."""
    normalized = norm(name)
    return next((code for code, aliases in DISTRICT_ALIASES.items() if normalized in {norm(alias) for alias in aliases}), None)

districts['district'] = districts.district_name.map(district_code)
districts = districts[districts.district.notna()][['district', 'geometry']]
if len(districts) != 4:
    raise ValueError('Could not identify all four selected alcaldías; inspect nomgeo values.')

risk_polygons = gpd.overlay(risk, districts, how='intersection', keep_geom_type=True)
risk_polygons = risk_polygons[~risk_polygons.geometry.is_empty].explode(index_parts=False, ignore_index=True)
risk_polygons = risk_polygons[risk_polygons.geometry.geom_type == 'Polygon'].reset_index(drop=True)
risk_polygons['risk_polygon_id'] = [f'R{i:05d}' for i in range(1, len(risk_polygons) + 1)]

def parcel_points(path, district):
    """Convert each valid cadastre parcel to one representative household-proxy point."""
    parcels = clean(read(path), path.name)
    parcels = parcels[parcels.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])].copy()
    return gpd.GeoDataFrame({'district': district, 'geometry': parcels.representative_point()}, geometry='geometry', crs=METRIC_CRS)

cadastre = gpd.GeoDataFrame(pd.concat([parcel_points(path, district) for district, path in cadastre_files.items()], ignore_index=True), geometry='geometry', crs=METRIC_CRS)
print('Selected risk categories:', sorted(selected_categories))
print('High-risk polygon records by district:', risk_polygons.groupby('district').size().to_dict())
print('Cadastre household-proxy points by district:', cadastre.groupby('district').size().to_dict())


/usr/local/lib/python3.13/dist-packages/pyogrio/raw.py:200: RuntimeWarning: Several features with id = 30 have been found. Altering it to be unique. This warning will not be emitted anymore for this layer
  return ogr_read(


Selected risk categories: ['alta']
High-risk polygon records by district: {'aob': 624, 'cuj': 637, 'gam': 168, 'izp': 70}
Cadastre household-proxy points by district: {'aob': 96448, 'cuj': 21786, 'gam': 163195, 'izp': 236303}


## Cluster polygons, create non-overlapping zones within each cluster, and count households

For `perimeter`, two polygons are connected when their buffered perimeters meet; this implements a minimum edge-to-edge gap. For `centroid`, their centroids are buffered instead. Connected components allow chains of qualifying polygons to become one cluster. Each completed cluster is dissolved before zones are built.

In [6]:
def connected_components_by_gap(polygons, gap_m, method):
    """Label polygons linked by a centroid or perimeter distance no greater than ``gap_m``.

    Buffering each comparison geometry by half the threshold means intersecting
    buffers are exactly equivalent to a pairwise distance of at most ``gap_m``.
    A union-find structure turns pairwise links into transitive cluster labels.
    """
    work = polygons.reset_index(drop=True).copy()
    comparison = work.geometry.centroid if method == 'centroid' else work.geometry
    expanded = gpd.GeoSeries(comparison.buffer(gap_m / 2), crs=work.crs)
    parent = list(range(len(work)))

    def find(item):
        while parent[item] != item:
            parent[item] = parent[parent[item]]
            item = parent[item]
        return item

    def union(left, right):
        left_root, right_root = find(left), find(right)
        if left_root != right_root:
            parent[right_root] = left_root

    pairs = expanded.sindex.query(expanded, predicate='intersects')
    for left, right in zip(pairs[0], pairs[1]):
        if left < right:
            union(int(left), int(right))

    component_roots = [find(index) for index in range(len(work))]
    root_to_number = {root: number for number, root in enumerate(sorted(set(component_roots)), start=1)}
    work['component_number'] = [root_to_number[root] for root in component_roots]
    return work

def make_clusters(polygons, gap_m, method):
    """Dissolve linked high-risk polygons and preserve each cluster's district membership."""
    labelled = connected_components_by_gap(polygons, gap_m, method)
    records = []
    for number, group in labelled.groupby('component_number', sort=True):
        records.append({
            'cluster_id': f'{method[:3].upper()}_{gap_m:03d}m_C{number:03d}',
            'cluster_gap_m': gap_m,
            'cluster_distance_method': method,
            'districts': ', '.join(sorted(group.district.unique())),
            'risk_polygon_count': len(group),
            'geometry': unary_union(group.geometry),
        })
    return gpd.GeoDataFrame(records, geometry='geometry', crs=polygons.crs)

def boundary_zones(cluster_geometry, inside_mode, inside_band_m, outside_band_m):
    """Return one inside and one outside boundary zone for a dissolved cluster.

    The outside ring is built after dissolving the cluster, preventing overlap
    among constituent polygons. It is not made exclusive across separate clusters.
    """
    boundary_buffer = cluster_geometry.boundary.buffer(max(inside_band_m, outside_band_m))
    inside = cluster_geometry if inside_mode == 'all_inside' else cluster_geometry.intersection(boundary_buffer)
    outside = cluster_geometry.boundary.buffer(outside_band_m).difference(cluster_geometry)
    return inside, outside

def count_points_once(points, spatial_index, zone):
    """Count representative points within one zone after spatial-index filtering."""
    candidate_positions = list(spatial_index.query(zone, predicate='intersects'))
    if not candidate_positions:
        return 0
    return int(points.iloc[candidate_positions].geometry.within(zone).sum())

cadastre_index = cadastre.sindex
clusters = make_clusters(risk_polygons, gap_m=CLUSTER_GAPS_M[0], method=CLUSTER_DISTANCE_METHOD)  # preview the first scenario
print(f'Preview: {len(clusters):,} clusters at {CLUSTER_GAPS_M[0]} m using {CLUSTER_DISTANCE_METHOD} distance.')


Preview: 486 clusters at 25 m using perimeter distance.


## Run all requested clustering gaps and export results

The GeoJSON contains cluster geometry plus its count fields. The CSV contains the same non-geometric table and is convenient for ranking candidate clusters. Review the map and physical setting before interpreting any counts as a usable quasi-experimental boundary.

In [7]:
scenario_results = []
for gap_m in CLUSTER_GAPS_M:
    clusters = make_clusters(risk_polygons, gap_m=gap_m, method=CLUSTER_DISTANCE_METHOD)
    inside_counts, outside_counts = [], []
    for cluster in clusters.itertuples():
        inside_zone, outside_zone = boundary_zones(
            cluster.geometry, INSIDE_MODE, INSIDE_BAND_M, OUTSIDE_BAND_M
        )
        inside_counts.append(count_points_once(cadastre, cadastre_index, inside_zone))
        outside_counts.append(count_points_once(cadastre, cadastre_index, outside_zone))
    clusters['inside_mode'] = INSIDE_MODE
    clusters['inside_band_m'] = np.nan if INSIDE_MODE == 'all_inside' else INSIDE_BAND_M
    clusters['outside_band_m'] = OUTSIDE_BAND_M
    clusters['cadastre_households_inside'] = inside_counts
    clusters['cadastre_households_outside'] = outside_counts
    clusters['inside_outside_balance_ratio'] = np.where(
        clusters.cadastre_households_outside > 0,
        clusters.cadastre_households_inside / clusters.cadastre_households_outside,
        np.nan,
    )
    scenario_results.append(clusters)

results = gpd.GeoDataFrame(pd.concat(scenario_results, ignore_index=True), geometry='geometry', crs=METRIC_CRS)
results['cluster_area_m2'] = results.geometry.area
results_wgs84 = results.to_crs(4326)
base_name = f'clustered_risk_household_counts_{CLUSTER_DISTANCE_METHOD}'
results_wgs84.to_file(WORK / f'{base_name}.geojson', driver='GeoJSON')
results.drop(columns='geometry').to_csv(WORK / f'{base_name}.csv', index=False)

display(
    results.drop(columns='geometry').sort_values(
        ['cluster_gap_m', 'cadastre_households_inside', 'cadastre_households_outside'],
        ascending=[True, False, False],
    )
)
print(f'Wrote {WORK / f"{base_name}.geojson"}')
print(f'Wrote {WORK / f"{base_name}.csv"}')


,cluster_id,cluster_gap_m,cluster_distance_method,districts,risk_polygon_count,inside_mode,inside_band_m,outside_band_m,cadastre_households_inside,cadastre_households_outside,inside_outside_balance_ratio,cluster_area_m2
12,PER_025m_C013,25,perimeter,aob,2,near_boundary,20,20,81,113,0.716814,35451.561038
447,PER_025m_C448,25,perimeter,gam,7,near_boundary,20,20,59,85,0.694118,65218.661646
22,PER_025m_C023,25,perimeter,aob,2,near_boundary,20,20,49,40,1.225000,14741.610561
63,PER_025m_C064,25,perimeter,aob,2,near_boundary,20,20,48,54,0.888889,28115.138164
437,PER_025m_C438,25,perimeter,gam,7,near_boundary,20,20,39,56,0.696429,527726.167440
...,...,...,...,...,...,...,...,...,...,...,...,...
1289,PER_200m_C135,200,perimeter,izp,2,near_boundary,20,20,0,0,NaN,2421.480895
1290,PER_200m_C136,200,perimeter,izp,2,near_boundary,20,20,0,0,NaN,2959.482244
1291,PER_200m_C137,200,perimeter,izp,6,near_boundary,20,20,0,0,NaN,33902.227740
1296,PER_200m_C142,200,perimeter,izp,3,near_boundary,20,20,0,0,NaN,18666.729258


Wrote /content/drive/MyDrive/Dissertation/data/output/clustered_risk_household_counts_perimeter.geojson
Wrote /content/drive/MyDrive/Dissertation/data/output/clustered_risk_household_counts_perimeter.csv
